# LeetCode 1660 — Correct a Binary Tree bằng Python

Link đề: https://leetcode.com/problems/correct-a-binary-tree/ · Độ khó: **Medium** · Chủ đề: Hash Table, Tree, Depth-First Search, Breadth-First Search, Binary Tree

> **Lưu ý (bài Premium 🔒):** đề trong notebook được viết lại theo hiểu biết về bài (nguồn tham khảo: bản mô tả của doocs/leetcode). Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def correctBinaryTree(self, root: TreeNode) -> TreeNode:
        ...
```

Cây có **đúng một nút lỗi**: con phải của nó trỏ **nhầm** sang một nút **cùng tầng, ở bên phải**. Ta phải xóa nút lỗi cùng toàn bộ cây con của nó (trừ nút bị trỏ nhầm). Notebook tự dựng **driver** gây lỗi y như đề mô tả, rồi đi từ **so sánh tầng** (vét cạn) → **BFS theo tầng** → **BFS từ phải sang trái** → **DFS ưu tiên nhánh phải** (đệ quy và lặp).

## 1. Mục tiêu học

1. Hiểu cấu trúc "cây có một cạnh thừa": nút `toNode` có **hai** đường đi tới (cha thật và nút lỗi), nhưng nút lỗi trỏ **ngang** (cùng tầng).
2. Nhận dạng nút lỗi bằng điều kiện: `u.right` **nằm cùng tầng** với `u` ⇔ khi duyệt **từ phải sang trái** trong một tầng, `u.right` **đã được thăm**.
3. Cài đặt BFS theo tầng (trái→phải và phải→trái) và DFS ưu tiên phải với tập `visited`; biết vì sao **thứ tự duyệt** quyết định đúng/sai.
4. Viết driver kiểu LeetCode: dựng cây từ level-order, gắn `fromNode.right = toNode`, gọi lời giải, xuất kết quả level-order.
5. Kiểm thử trên cây ngẫu nhiên với lỗi được gây có chủ đích, so với "đáp án chuẩn" tính từ cây gốc.

## 2. Đề bài & ví dụ

Có một cây nhị phân với **một khuyết tật nhỏ**: có **đúng một** nút không hợp lệ, con phải của nó **trỏ nhầm** tới một nút khác **cùng độ sâu** và nằm **bên phải** nút đó.

Cho gốc `root` của cây bị lỗi, trả về gốc của cây sau khi **xóa nút không hợp lệ và mọi nút bên dưới nó** (trừ nút mà nó trỏ nhầm tới).

**Kiểm thử tùy chỉnh (custom testing):** đầu vào gồm 3 dòng: cây `root`, `fromNode`, `toNode` (hai số này **không** được truyền cho `correctBinaryTree`). Sau khi dựng cây, nút có giá trị `fromNode` bị gán `right` trỏ tới nút có giá trị `toNode`, rồi mới truyền `root` cho hàm.

**Ví dụ 1:** `root = [1,2,3]`, `fromNode = 2`, `toNode = 3` → `[1,null,3]`. Nút 2 lỗi, xóa nó.

**Ví dụ 2:** `root = [8,3,1,7,null,9,4,2,null,null,null,5,6]`, `fromNode = 7`, `toNode = 4` → `[8,3,1,null,null,9,4,null,null,5,6]`. Nút 7 lỗi, xóa nó và nút bên dưới nó (nút 2).

```
Ví dụ 2 trước khi sửa (mũi tên ⇢ là con trỏ lỗi 7.right = 4):

              8
           /     \
          3       1
         /       / \
        7  ⇢⇢⇢  9   4
       /           / \
      2           5   6
```

**Ràng buộc:**
- Số nút trong khoảng `[3, 10⁴]`; giá trị khác nhau trong `[-10⁹, 10⁹]`.
- `fromNode != toNode`; cả hai tồn tại và **cùng độ sâu**; `toNode` ở **bên phải** `fromNode`.
- `fromNode.right` là `null` trong cây ban đầu của dữ liệu test.

In [1]:
import random
import sys
import time
from collections import deque
from typing import Optional, List

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def __repr__(self):
        return f"TreeNode({self.val})"

def build_tree(values):
    """Dựng cây từ level-order kiểu LeetCode; trả về (root, dict val -> node)."""
    if not values or values[0] is None:
        return None, {}
    root = TreeNode(values[0]); index = {root.val: root}
    q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()
        for side in ("left", "right"):
            if i < len(values) and values[i] is not None:
                child = TreeNode(values[i]); setattr(node, side, child)
                index[child.val] = child; q.append(child)
            i += 1
    return root, index

def to_list(root):
    """Cây hợp lệ -> level-order (bỏ None ở cuối)."""
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val); q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def make_defect(values, from_val, to_val):
    """Driver như LeetCode: dựng cây rồi gán from.right = to."""
    root, index = build_tree(values)
    assert index[from_val].right is None, "đề bảo đảm fromNode.right ban đầu là null"
    index[from_val].right = index[to_val]
    return root

examples = [
    ([1, 2, 3], 2, 3, [1, None, 3]),
    ([8, 3, 1, 7, None, 9, 4, 2, None, None, None, 5, 6], 7, 4, [8, 3, 1, None, None, 9, 4, None, None, 5, 6]),
]
r = make_defect(*examples[1][:3])
print("Sau khi gây lỗi: 7.right =", r.left.left.right, "| cha thật của 4 là 1:", r.right.right)

Sau khi gây lỗi: 7.right = TreeNode(4) | cha thật của 4 là 1: TreeNode(4)


## 3. Kiến thức nền

### 3.1 Duyệt theo tầng (BFS level-order)

Mỗi vòng ngoài lấy hết các nút của **một tầng** (độ dài hàng đợi lúc bắt đầu vòng). Thứ tự `left` rồi `right` cho tầng từ trái sang phải; đảo lại `right` rồi `left` cho tầng **từ phải sang trái**.

In [2]:
def levels(root, right_to_left=False):
    out, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            kids = (node.right, node.left) if right_to_left else (node.left, node.right)
            for k in kids:
                if k: q.append(k)
        out.append(level)
    return out

root, _ = build_tree(examples[1][0])
print("trái -> phải:", levels(root))
print("phải -> trái:", levels(root, right_to_left=True))

trái -> phải: [[8], [3, 1], [7, 9, 4], [2, 5, 6]]
phải -> trái: [[8], [1, 3], [4, 9, 7], [6, 5, 2]]


### 3.2 Cây có một cạnh thừa — đừng duyệt "ngây thơ"

Sau khi gây lỗi, nút `toNode` có **hai** con trỏ đi vào. Một BFS/DFS không có `visited` sẽ thăm `toNode` (và cả cây con của nó) **hai lần**. Ta in số lần thăm để thấy rõ:

In [3]:
def count_visits(root):
    cnt = {}
    q = deque([root])
    while q:
        node = q.popleft()
        cnt[node.val] = cnt.get(node.val, 0) + 1
        for k in (node.left, node.right):
            if k: q.append(k)
    return cnt

bad_root = make_defect(*examples[1][:3])
print({v: c for v, c in count_visits(bad_root).items() if c > 1}, "<- các nút bị thăm 2 lần")

{4: 2, 5: 2, 6: 2} <- các nút bị thăm 2 lần


### 3.3 Tập `visited` theo định danh nút

`TreeNode` mặc định băm theo `id`, nên `set()` các nút hoạt động tốt; `node in seen` là `O(1)`.

In [4]:
a, b = TreeNode(1), TreeNode(1)
seen = {a}
print(a in seen, b in seen, "(cùng giá trị nhưng khác đối tượng)")

True False (cùng giá trị nhưng khác đối tượng)


### 3.4 Xóa một cây con

Muốn xóa nút `u` cùng cây con, chỉ cần **cắt con trỏ từ cha**: `parent.left = None` (hoặc `.right`). Bộ thu gom rác của Python tự dọn phần còn lại (miễn là không còn tham chiếu khác).

In [5]:
root, idx = build_tree([1, 2, 3, 4])
print("trước:", to_list(root))
idx[1].left = None              # xóa nút 2 và cây con (gồm 4)
print("sau  :", to_list(root))

trước: [1, 2, 3, 4]
sau  : [1, None, 3]


## 4. Ý tưởng nền tảng / trực giác

Với ví dụ 2, các tầng của cây **lỗi** (đi theo cả con trỏ lỗi):
- Tầng 2 (từ trái sang phải): `7, 9, 4`. Nút `7` có `right = 4` — mà `4` **cũng ở tầng 2**! Đây là dấu hiệu: một nút bình thường có con ở tầng **dưới**, chỉ nút lỗi mới có "con" ở **cùng tầng**.

Vì `toNode` nằm **bên phải** `fromNode`, nếu duyệt mỗi tầng **từ phải sang trái** thì ta gặp `toNode` **trước** `fromNode`. Khi tới `fromNode`, con phải của nó **đã được thăm** trong chính tầng này ⇒ phát hiện ngay.

Sau khi tìm được nút lỗi `u` cùng cha `p` của nó: gán `p.left = None` hoặc `p.right = None` (tùy `u` là con nào). Đáp án ví dụ 2: xóa `7` (và `2`) ⇒ `[8,3,1,null,null,9,4,null,null,5,6]`.

In [6]:
bad_root = make_defect(*examples[1][:3])
q = deque([bad_root]); depth = 0; seen = {bad_root}
while q:
    level = list(q)
    print(f"tầng {depth}: {[n.val for n in level]}", end="")
    same = [n.val for n in level if n.right in set(level)]
    print(f"  -> nút có right cùng tầng: {same}" if same else "")
    nxt = deque()
    for n in level:
        for k in (n.left, n.right):
            if k and k not in seen and k not in set(level):
                seen.add(k); nxt.append(k)
    q = nxt; depth += 1

tầng 0: [8]
tầng 1: [3, 1]
tầng 2: [7, 9, 4]  -> nút có right cùng tầng: [7]
tầng 3: [2, 5, 6]


## 5. Cách 1 — Brute Force / baseline: tính tầng của mọi nút, kiểm tra từng nút

**Ý tưởng:**
1. Tính **độ sâu thật** của mọi nút bằng BFS (lần đầu gặp một nút là theo đường ngắn nhất ⇒ `toNode` nhận độ sâu đúng qua cha thật, vì đường qua nút lỗi dài hơn 1).
2. Với **mỗi** nút `u`, duyệt **danh sách** các nút cùng tầng xem `u.right` có trong đó không (dùng list để thể hiện cách "ngây thơ").
3. Tìm cha của nút lỗi và cắt.

In [7]:
def correct_brute(root):
    depth = {root: 0}
    parent = {root: None}
    q = deque([root])
    while q:
        u = q.popleft()
        for side in ("left", "right"):
            k = getattr(u, side)
            if k and k not in depth:
                depth[k] = depth[u] + 1
                parent[k] = (u, side)
                q.append(k)
    by_level = {}
    for node, d in depth.items():
        by_level.setdefault(d, []).append(node)       # list: tra cứu O(độ rộng tầng)
    for u, d in depth.items():
        if u.right is not None and u.right in by_level[d]:
            p, side = parent[u]
            setattr(p, side, None)
            return root
    return root

for vals, f, t, exp in examples:
    got = to_list(correct_brute(make_defect(vals, f, t)))
    print(got, "| đúng" if got == exp else "| SAI")

[1, None, 3] | đúng
[8, 3, 1, None, None, 9, 4, None, None, 5, 6] | đúng


**Độ phức tạp:** BFS `O(n)`; bước kiểm tra là `n` lần tra trong list cỡ độ rộng tầng ⇒ `O(n · w)`, xấu nhất `O(n²)` (cây rất rộng). Bộ nhớ `O(n)` cho `depth`, `parent`.

**Nút thắt:** tra cứu trong list chậm, và ta duyệt cả cây hai lần. Hơn nữa, có thể phát hiện nút lỗi **ngay trong lúc BFS** thay vì làm hậu kỳ.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** trong cây hợp lệ, `u.right` (nếu có) ở tầng `depth(u) + 1`. Nút lỗi là nút **duy nhất** có `u.right` ở **cùng tầng** `depth(u)`.

**Quy tắc khi duyệt một tầng từ phải sang trái** (với tập `seen` gồm các nút đã thăm):
- **Nếu** `u.right` đã nằm trong `seen` **thì** `u` là nút lỗi (vì `u.right` cùng tầng và ở bên phải ⇒ đã được thăm trước `u`) ⇒ cắt `u` khỏi cha.
- **Ngược lại**, đưa các con của `u` vào tầng sau (theo thứ tự phải trước, trái sau).

**Thứ tự duyệt quan trọng tới đâu?**
- Với **DFS**: bắt buộc **phải trước** — nếu trái trước, ta tới `fromNode` khi `toNode` chưa được thăm (lỗi 1, mục 13).
- Với **BFS đánh dấu `seen` lúc lấy ra** khỏi hàng đợi: bắt buộc **phải → trái** để `toNode` được lấy ra trước `fromNode` (lỗi 3, mục 13).
- Với **BFS đánh dấu `seen` lúc đưa vào**: `toNode` đã được đánh dấu khi cha thật của nó (tầng trên) được xử lý, nên thứ tự trong tầng không còn quan trọng. Ta vẫn chọn **phải → trái** cho an toàn — lời giải đúng với cả hai kiểu đánh dấu.

**DFS tương đương:** duyệt **phải trước, trái sau** (tiền thứ tự đảo). Mọi nút bên phải trên cùng tầng của `u` được thăm trước `u`; khi tới `u` mà `u.right` đã thăm ⇒ `u` là nút lỗi ⇒ trả về `None` để cha "cắt" nó.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS theo tầng, kiểm tra bằng tập nút của tầng (trái → phải)

Mỗi tầng: tạo `level_set` gồm **mọi nút trong tầng** trước, rồi duyệt từng nút `u` và kiểm tra `u.right in level_set`. Vì cả tầng đã biết trước nên thứ tự duyệt không còn quan trọng — nhưng phải cẩn thận **không** đẩy `u.right` của nút lỗi vào tầng sau.

In [8]:
def correct_bfs_levelset(root):
    level = [(root, None, None)]            # (nút, cha, phía)
    while level:
        level_set = {node for node, _, _ in level}
        nxt = []
        for node, par, side in level:
            if node.right is not None and node.right in level_set:
                setattr(par, side, None)    # nút lỗi: cắt khỏi cha
                return root
            for s in ("left", "right"):
                k = getattr(node, s)
                if k:
                    nxt.append((k, node, s))
        level = nxt
    return root

for vals, f, t, exp in examples:
    got = to_list(correct_bfs_levelset(make_defect(vals, f, t)))
    print(got, "| đúng" if got == exp else "| SAI")

[1, None, 3] | đúng
[8, 3, 1, None, None, 9, 4, None, None, 5, 6] | đúng


Lưu ý: trước khi tới tầng của nút lỗi, cây phía trên hoàn toàn hợp lệ nên `nxt` không có trùng lặp. Ta dừng ngay ở tầng chứa nút lỗi. **Độ phức tạp:** `O(n)` thời gian, `O(w)` bộ nhớ (độ rộng tầng lớn nhất).

### 7.2 Cách 3 — BFS từ phải sang trái với tập `seen`

In [9]:
def correct_bfs_rtl(root):
    seen = {root}
    q = deque([(root, None, None)])
    while q:
        for _ in range(len(q)):
            node, par, side = q.popleft()
            if node.right is not None and node.right in seen:
                setattr(par, side, None)
                return root
            for s in ("right", "left"):            # phải trước, trái sau
                k = getattr(node, s)
                if k and k not in seen:
                    seen.add(k)
                    q.append((k, node, s))
    return root

class SolutionBFS:
    def correctBinaryTree(self, root: TreeNode) -> TreeNode:
        seen = {root}
        q = deque([(root, None, None)])
        while q:
            node, par, side = q.popleft()
            if node.right is not None and node.right in seen:
                setattr(par, side, None)
                return root
            for s in ("right", "left"):
                k = getattr(node, s)
                if k is not None and k not in seen:
                    seen.add(k)
                    q.append((k, node, s))
        return root

for vals, f, t, exp in examples:
    a = to_list(correct_bfs_rtl(make_defect(vals, f, t)))
    b = to_list(SolutionBFS().correctBinaryTree(make_defect(vals, f, t)))
    print(a, b, "| đúng" if a == b == exp else "| SAI")

[1, None, 3] [1, None, 3] | đúng
[8, 3, 1, None, None, 9, 4, None, None, 5, 6] [8, 3, 1, None, None, 9, 4, None, None, 5, 6] | đúng


Vì đánh dấu `seen` **khi đưa vào** hàng đợi, `toNode` đã nằm trong `seen` từ khi cha thật của nó được xử lý ở tầng trên. Khi tới `fromNode`, `fromNode.right in seen` ⇒ phát hiện. Còn với nút hợp lệ `u`, `u.right` chỉ được đưa vào `seen` khi chính `u` được xử lý — tức **sau** lúc kiểm tra. **Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.3 Cách 4 — DFS đệ quy ưu tiên nhánh phải

In [10]:
def correct_dfs(root):
    seen = set()
    def dfs(node):
        if node is None:
            return None
        if node.right in seen:            # con phải đã thăm => node lỗi
            return None
        seen.add(node)
        node.right = dfs(node.right)      # PHẢI trước
        node.left = dfs(node.left)
        return node
    return dfs(root)

for vals, f, t, exp in examples:
    got = to_list(correct_dfs(make_defect(vals, f, t)))
    print(got, "| đúng" if got == exp else "| SAI")

[1, None, 3] | đúng
[8, 3, 1, None, None, 9, 4, None, None, 5, 6] | đúng


Điều kiện `node.right in seen` với `node.right = None` luôn sai (vì `None` không nằm trong `seen`) nên không cần kiểm tra riêng. **Độ phức tạp:** `O(n)` thời gian, `O(h)` stack đệ quy — với cây lệch `10⁴` nút sẽ vượt giới hạn đệ quy mặc định của Python (mục 13).

### 7.4 Cách 5 — DFS lặp ưu tiên nhánh phải (stack)

Mô phỏng tiền thứ tự "phải trước": đẩy **trái** vào stack trước rồi **phải** sau (để phải được lấy ra trước). Mang theo `(cha, phía)` để cắt khi gặp nút lỗi.

In [11]:
def correct_dfs_iter(root):
    seen = set()
    stack = [(root, None, None)]
    while stack:
        node, par, side = stack.pop()
        if node.right is not None and node.right in seen:
            setattr(par, side, None)
            return root
        seen.add(node)
        if node.left:
            stack.append((node.left, node, "left"))
        if node.right:
            stack.append((node.right, node, "right"))     # lấy ra trước
    return root

for vals, f, t, exp in examples:
    got = to_list(correct_dfs_iter(make_defect(vals, f, t)))
    print(got, "| đúng" if got == exp else "| SAI")

[1, None, 3] | đúng
[8, 3, 1, None, None, 9, 4, None, None, 5, 6] | đúng


**Vì sao vẫn đúng?** Thứ tự lấy ra là tiền thứ tự **phải–trái** (gốc, cây con phải, cây con trái). Với hai nút cùng tầng `A` (trái) và `B` (phải), tổ tiên chung thấp nhất của chúng có `B` ở nhánh phải ⇒ `B` được thăm trước `A`. Vậy khi tới `fromNode`, `toNode` đã ở trong `seen`. Ngược lại, với nút hợp lệ `u`, `u.right` là con của `u` nên **chưa** được thăm khi xử lý `u`. **Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ, không đệ quy.

## 8. TRACE từng bước

Trace cách 3 (BFS phải → trái) trên ví dụ 2.

| tầng | thứ tự xử lý | kiểm tra `right in seen` | seen sau tầng |
|---|---|---|---|
| 0 | 8 | 1? chưa | {8, 1, 3} |
| 1 | 1, 3 | 4? chưa; (3.right=None) | + {4, 9, 7} |
| 2 | 4, 9, **7** | 6? chưa; None; **7.right = 4 ∈ seen** ⇒ lỗi | dừng, cắt 3.left |

In [12]:
def trace_bfs_rtl(values, f, t):
    root = make_defect(values, f, t)
    seen = {root}
    q = deque([(root, None, None)])
    depth = 0
    while q:
        print(f"tầng {depth}: hàng đợi = {[n.val for n, _, _ in q]}")
        for _ in range(len(q)):
            node, par, side = q.popleft()
            r = node.right
            if r is not None and r in seen:
                print(f"   xử lý {node.val}: right = {r.val} ĐÃ THĂM -> {node.val} là nút lỗi; cắt {par.val}.{side}")
                setattr(par, side, None)
                print("   kết quả:", to_list(root))
                return root
            added = []
            for s in ("right", "left"):
                k = getattr(node, s)
                if k and k not in seen:
                    seen.add(k); q.append((k, node, s)); added.append(k.val)
            print(f"   xử lý {node.val}: right = {r.val if r else None} chưa thăm -> thêm {added}")
        depth += 1

trace_bfs_rtl(*examples[1][:3])

tầng 0: hàng đợi = [8]
   xử lý 8: right = 1 chưa thăm -> thêm [1, 3]
tầng 1: hàng đợi = [1, 3]
   xử lý 1: right = 4 chưa thăm -> thêm [4, 9]
   xử lý 3: right = None chưa thăm -> thêm [7]
tầng 2: hàng đợi = [4, 9, 7]
   xử lý 4: right = 6 chưa thăm -> thêm [6, 5]
   xử lý 9: right = None chưa thăm -> thêm []
   xử lý 7: right = 4 ĐÃ THĂM -> 7 là nút lỗi; cắt 3.left
   kết quả: [8, 3, 1, None, None, 9, 4, None, None, 5, 6]


TreeNode(8)

Trace cách 4 (DFS phải trước) trên ví dụ 2 — in thứ tự thăm:

In [13]:
def trace_dfs(values, f, t):
    root = make_defect(values, f, t)
    seen = set()
    def dfs(node, depth):
        if node is None:
            return None
        if node.right in seen:
            print("  " * depth + f"{node.val}: right={node.right.val} đã thăm -> trả None (xóa {node.val})")
            return None
        seen.add(node)
        print("  " * depth + f"thăm {node.val}")
        node.right = dfs(node.right, depth + 1)
        node.left = dfs(node.left, depth + 1)
        return node
    dfs(root, 0)
    print("kết quả:", to_list(root))

trace_dfs(*examples[1][:3])

thăm 8
  thăm 1
    thăm 4
      thăm 6
      thăm 5
    thăm 9
  thăm 3
    7: right=4 đã thăm -> trả None (xóa 7)
kết quả: [8, 3, 1, None, None, 9, 4, None, None, 5, 6]


## 9. Code nộp LeetCode

Lời giải chính: **BFS phải → trái** (lặp, không đệ quy, an toàn với cây lệch `10⁴` nút).

In [14]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def correctBinaryTree(self, root: TreeNode) -> TreeNode:
        seen = {root}
        q = deque([(root, None, None)])            # (nút, cha, 'left'/'right')
        while q:
            node, par, side = q.popleft()
            if node.right is not None and node.right in seen:
                setattr(par, side, None)           # xóa nút lỗi và cây con của nó
                return root
            for s in ("right", "left"):            # phải trước để toNode được thăm trước fromNode
                child = getattr(node, s)
                if child is not None and child not in seen:
                    seen.add(child)
                    q.append((child, node, s))
        return root

for vals, f, t, exp in examples:
    assert to_list(Solution().correctBinaryTree(make_defect(vals, f, t))) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


Nút lỗi không bao giờ là gốc (gốc ở tầng 0 không có nút nào cùng tầng bên phải), nên `par` luôn khác `None` khi cắt.

## 10. Vì sao thuật toán đúng

Gọi `F` = nút lỗi, `T` = nút bị trỏ nhầm, `d` = tầng chung của chúng. Bỏ cạnh lỗi `F → T` đi, ta được cây hợp lệ ban đầu `G`.

1. **Không báo nhầm trước tầng `d`:** trên các tầng `< d`, mọi con trỏ đều là cạnh của `G`, nên `u.right` là con thật của `u`, nằm ở tầng sau và **chưa** được đưa vào `seen` khi xử lý `u` (chỉ `u` mới đưa nó vào). Ở tầng `d`, các nút hợp lệ cũng vậy.
2. **Phát hiện đúng `F`:** BFS xử lý tầng `d − 1` từ phải sang trái và đưa con vào hàng đợi theo thứ tự phải → trái, nên tầng `d` cũng được xử lý từ phải sang trái và mọi nút tầng `d` đã nằm trong `seen` khi tầng `d` bắt đầu. Vì `T` là một nút tầng `d` (thông qua cha thật), khi xử lý `F` thì `F.right = T ∈ seen` ⇒ điều kiện đúng. (Trước `F`, không nút tầng `d` nào thỏa điều kiện theo (1).)
3. **Kết quả đúng:** cắt `F` khỏi cha xóa đúng cây con của `F` trong `G` (gồm `F.left` và các hậu duệ). `T` vẫn được giữ qua cha thật. Đó chính là cây đề yêu cầu.

**Cách DFS (4, 5):** tiền thứ tự phải–trái thăm `T` trước `F` (lập luận qua LCA ở mục 7.4); với nút hợp lệ `u`, `u.right` là con của `u` nên chưa thăm. Khi `dfs(F)` trả `None`, cha của `F` gán con trỏ tương ứng về `None` — tức cắt `F`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h` là chiều cao, `w` là độ rộng lớn nhất của một tầng.

| Cách | Time | Space |
|---|---|---|
| 1. Tầng + list | `O(n · w)` (xấu nhất `O(n²)`) | `O(n)` |
| 2. BFS + tập nút của tầng | `O(n)` | `O(w)` |
| 3. BFS phải → trái + `seen` | `O(n)` | `O(n)` |
| 4. DFS đệ quy phải trước | `O(n)` | `O(n)` (`seen`) + `O(h)` stack |
| 5. DFS lặp phải trước | `O(n)` | `O(n)` |

Đo trên cây ngẫu nhiên `10⁴` nút và cây "lệch" sâu (bản đệ quy chỉ chạy trên cây ngẫu nhiên):

In [15]:
def random_tree_values(n, rnd):
    """Sinh cây ngẫu nhiên n nút (giá trị 1..n) và trả về level-order list."""
    vals = list(range(1, n + 1)); rnd.shuffle(vals)
    root = TreeNode(vals[0]); slots = [(root, 'left'), (root, 'right')]
    for v in vals[1:]:
        k = rnd.randrange(len(slots)); par, side = slots[k]
        slots[k] = slots[-1]; slots.pop()
        ch = TreeNode(v); setattr(par, side, ch)
        slots += [(ch, 'left'), (ch, 'right')]
    return to_list(root)

def pick_defect(values, rnd):
    """Chọn (from, to) hợp lệ: cùng tầng, to bên phải from, from.right == None. None nếu không có."""
    root, idx = build_tree(values)
    cands = []
    q = [root]
    while q:
        for i, u in enumerate(q):
            if u.right is None:
                for v in q[i + 1:]:
                    cands.append((u.val, v.val))
        q = [k for u in q for k in (u.left, u.right) if k]
    return rnd.choice(cands) if cands else None

rnd = random.Random(1660)
vals = random_tree_values(10**4, rnd)
f, t = pick_defect(vals, rnd)
for name, fn in [("brute", correct_brute), ("levelset", correct_bfs_levelset), ("bfs_rtl", correct_bfs_rtl), ("dfs_rec", correct_dfs), ("dfs_iter", correct_dfs_iter), ("Solution", Solution().correctBinaryTree)]:
    rt = make_defect(vals, f, t)
    t0 = time.perf_counter(); fn(rt); dt = time.perf_counter() - t0
    print(f"{name:<9}: {dt * 1000:6.1f} ms, còn {len([v for v in to_list(rt) if v is not None])} nút")

brute    :    6.5 ms, còn 9999 nút
levelset :    0.4 ms, còn 9999 nút
bfs_rtl  :    0.5 ms, còn 9999 nút
dfs_rec  :    1.1 ms, còn 9999 nút
dfs_iter :    0.4 ms, còn 9999 nút
Solution :    0.5 ms, còn 9999 nút


## 12. Bộ test

**Đáp án chuẩn** cho một bộ `(cây, from, to)`: lấy cây **gốc** (chưa lỗi), xóa cây con của `from` ⇒ level-order. Ta so mọi cách với đáp án chuẩn trên: ví dụ đề, các edge case (lỗi ở tầng sâu nhất, `from` là con trái/phải của cha, `from` và `to` khác cha / cùng cha, `from` có cây con lớn), và nhiều cây ngẫu nhiên có seed.

In [16]:
def expected_after_fix(values, from_val):
    root, idx = build_tree(values)
    # tìm cha của from và cắt
    q = deque([root])
    while q:
        u = q.popleft()
        for side in ("left", "right"):
            k = getattr(u, side)
            if k is not None:
                if k.val == from_val:
                    setattr(u, side, None)
                    return to_list(root)
                q.append(k)
    raise ValueError("from là gốc?")

ALL = {
    "brute": correct_brute,
    "levelset": correct_bfs_levelset,
    "bfs_rtl": correct_bfs_rtl,
    "dfs_rec": correct_dfs,
    "dfs_iter": correct_dfs_iter,
    "SolBFS": lambda r: SolutionBFS().correctBinaryTree(r),
    "Solution": lambda r: Solution().correctBinaryTree(r),
}

def check(values, f, t, exp=None, skip=()):
    want = expected_after_fix(values, f)
    if exp is not None:
        assert want == exp, (want, exp)
    for name, fn in ALL.items():
        if name in skip:
            continue
        got = to_list(fn(make_defect(values, f, t)))
        assert got == want, (name, values, f, t, got, want)
    return want

for vals_, f, t, exp in examples:
    check(vals_, f, t, exp)
print("Ví dụ đề: OK")

check([1, 2, 3, 4, None, 5, 6], 4, 6)          # from là con trái, to ở cha khác
check([1, 2, 3, 4, None, 5, 6], 4, 5)          # to là con trái của cha khác
check([1, 2, 3, None, 4, 5, None], 4, 5)       # from là con phải
check([1, 2, 3, 4, 5, 6, 7, 8], 6, 7)          # from và to cùng cha (6.right ban đầu None)
check([1, 2, 3, 4, None, None, 5, 6, 7], 2, 3) # from có cây con lớn
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [17]:
rnd = random.Random(2016)
done = 0
while done < 1500:
    n = rnd.randint(3, 25)
    vals_ = random_tree_values(n, rnd)
    pick = pick_defect(vals_, rnd)
    if pick is None:
        continue
    check(vals_, *pick)
    done += 1
print("1500 cây ngẫu nhiên nhỏ có lỗi gây chủ đích: OK")

# cây sâu: một "xương sống" dài 6000 nút bên trái, mỗi nút có thêm con phải là lá
def spine_values(m):
    root = TreeNode(1); cur = root; v = 2
    for _ in range(m):
        cur.right = TreeNode(v); v += 1
        cur.left = TreeNode(v); v += 1
        cur = cur.left
    return to_list(root)

sv = spine_values(3000)
rt, idx = build_tree(sv)
# chọn tầng sâu: nút lá trái cuối cùng và lá phải cùng tầng
deep = [rt]
while True:
    nxt = [k for u in deep for k in (u.left, u.right) if k]
    if not nxt: break
    deep = nxt
f_node, t_node = deep[0], deep[1]
check(sv, f_node.val, t_node.val, skip=("dfs_rec", "brute"))
print("Cây sâu ~6000 nút, lỗi ở tầng cuối (các bản lặp): OK")

1500 cây ngẫu nhiên nhỏ có lỗi gây chủ đích: OK


Cây sâu ~6000 nút, lỗi ở tầng cuối (các bản lặp): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — DFS ưu tiên nhánh **trái**

In [18]:
def correct_bug_dfs_left(root):
    seen = set()
    def dfs(node):
        if node is None or node.right in seen:
            return None
        seen.add(node)
        node.left = dfs(node.left)        # SAI: trái trước
        node.right = dfs(node.right)
        return node
    return dfs(root)

vals_, f, t, exp = examples[1]
print("trái trước:", to_list(correct_bug_dfs_left(make_defect(vals_, f, t))))
print("đúng       :", exp)

trái trước: [8, 3, None, 7, None, 2, 4, None, None, 5, 6]
đúng       : [8, 3, 1, None, None, 9, 4, None, None, 5, 6]


**Vì sao sai:** đi trái trước thì gặp `7` khi `4` **chưa** được thăm ⇒ không phát hiện; DFS còn đi theo con trỏ lỗi `7 → 4`, thăm `4` như con của `7`. Về sau, khi tới nút **hợp lệ** `1`, thấy `1.right = 4` đã thăm ⇒ coi nhầm `1` là nút lỗi và **xóa `1` (cùng `9`)**, còn `7` ở lại và mang theo `4`. **Sửa:** duyệt **phải trước**.

### Lỗi 2 — Chỉ xóa con trỏ lỗi (`from.right = None`) mà không xóa nút lỗi

In [19]:
def correct_bug_only_pointer(root):
    seen = {root}
    q = deque([root])
    while q:
        node = q.popleft()
        if node.right is not None and node.right in seen:
            node.right = None                  # SAI: nút lỗi vẫn còn trong cây
            return root
        for s in ("right", "left"):
            k = getattr(node, s)
            if k and k not in seen:
                seen.add(k); q.append(k)
    return root

vals_, f, t, exp = examples[1]
print("chỉ cắt con trỏ:", to_list(correct_bug_only_pointer(make_defect(vals_, f, t))))
print("đúng           :", exp)

chỉ cắt con trỏ: [8, 3, 1, 7, None, 9, 4, 2, None, None, None, 5, 6]
đúng           : [8, 3, 1, None, None, 9, 4, None, None, 5, 6]


**Vì sao sai:** đề yêu cầu xóa **nút lỗi và mọi nút bên dưới nó** (`7` và `2`). Chỉ cắt `7.right` thì cây vẫn chứa `7`, `2`. **Sửa:** mang theo `(cha, phía)` và cắt `cha.side = None`.

### Lỗi 3 — BFS trái → phải, đánh dấu `seen` lúc **lấy ra**

In [20]:
def correct_bug_bfs_ltr_pop(root):
    seen = set()
    q = deque([(root, None, None)])
    while q:
        node, par, side = q.popleft()
        if node in seen:
            continue
        if node.right is not None and node.right in seen:
            setattr(par, side, None)
            return root
        seen.add(node)                         # đánh dấu lúc LẤY RA
        for s in ("left", "right"):            # SAI kết hợp: trái trước
            k = getattr(node, s)
            if k:
                q.append((k, node, s))
    return root

vals_, f, t, exp = examples[1]
print("BFS trái->phải, đánh dấu lúc lấy ra:", to_list(correct_bug_bfs_ltr_pop(make_defect(vals_, f, t))))
print("đúng                               :", exp)
print("ví dụ 1:", to_list(correct_bug_bfs_ltr_pop(make_defect(*examples[0][:3]))), "| đúng:", examples[0][3])

BFS trái->phải, đánh dấu lúc lấy ra: [8, 3, 1, 7, None, 9, 4, 2, 4, None, None, 5, 6, None, None, 5, 6]
đúng                               : [8, 3, 1, None, None, 9, 4, None, None, 5, 6]
ví dụ 1: [1, 2, 3, None, 3] | đúng: [1, None, 3]


**Vì sao sai:** ở tầng 2 thứ tự lấy ra là `7, 9, 4`; khi xét `7`, nút `4` **chưa được lấy ra** nên chưa có trong `seen` ⇒ không phát hiện, và con trỏ lỗi vẫn nằm nguyên (kết quả in ra còn `4` xuất hiện hai lần — thực chất là cây chưa được sửa). **Sửa:** duyệt phải → trái, hoặc đánh dấu `seen` ngay khi đưa vào hàng đợi (như lời giải nộp — làm cả hai cho chắc chắn).

### Lỗi 4 — Đệ quy trên cây sâu

In [21]:
rt, _ = build_tree(sv)
deep = [rt]
while True:
    nxt = [k for u in deep for k in (u.left, u.right) if k]
    if not nxt: break
    deep = nxt
deep[0].right = deep[1]
try:
    correct_dfs(rt)
    print("Đệ quy chạy được (giới hạn đệ quy đủ lớn)")
except RecursionError:
    print("RecursionError trên cây sâu ~3000 tầng (giới hạn =", sys.getrecursionlimit(), ")")
rt2 = make_defect(sv, deep[0].val, deep[1].val)
print("Bản lặp:", "OK" if to_list(Solution().correctBinaryTree(rt2)) == expected_after_fix(sv, deep[0].val) else "SAI")

RecursionError trên cây sâu ~3000 tầng (giới hạn = 1000 )
Bản lặp: OK


**Sửa:** dùng BFS hoặc DFS lặp. (Lưu ý thêm: `to_list` cũng phải lặp, không đệ quy, nếu muốn in cây sâu.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Con trỏ trỏ **ngang** sang nút cùng tầng bên phải" ⇒ duyệt **từ phải sang trái** (DFS phải trước, BFS phải → trái): nút bị trỏ tới đã thăm trước nút lỗi.
- Cây có thêm một cạnh (thành DAG) ⇒ luôn dùng `visited` để không thăm hai lần.
- Muốn **xóa** một nút trong cây ⇒ cần biết **cha** và **phía**; mang theo `(parent, side)` trong hàng đợi/stack, hoặc để hàm đệ quy **trả về nút mới** cho cha gán lại (`node.right = dfs(node.right)`).
- Bài cây `10⁴` nút trong Python ⇒ ưu tiên lời giải lặp.
- Tự viết **driver gây lỗi** + **đáp án chuẩn từ cây gốc** để test — rất quan trọng với bài custom-testing.

## 15. Biến thể của bài

1. **Con trỏ lỗi trỏ sang nút cùng tầng bên trái:** đổi thứ tự duyệt thành trái → phải.
2. **Con trỏ lỗi là `left` thay vì `right`:** kiểm tra `node.left in seen` (và chú ý thứ tự).
3. **Con trỏ lỗi trỏ lên tổ tiên (tạo chu trình):** DFS với `visited` / màu đỉnh để phát hiện cạnh ngược.
4. **Chỉ cần tìm nút lỗi (không xóa):** trả về `node` thay vì cắt.
5. **LeetCode 116/117 — Populating Next Right Pointers:** cũng thao tác "nút cùng tầng bên phải" bằng BFS theo tầng.
6. **Nhiều nút lỗi** (mỗi nút trỏ sang phải cùng tầng): BFS phải → trái vẫn phát hiện được tất cả nếu không dừng sớm (cần cẩn thận khi nút lỗi nằm trong cây con của nút lỗi khác).

In [22]:
def find_invalid(root):
    """Biến thể 4: trả về giá trị nút lỗi."""
    seen = {root}; q = deque([root])
    while q:
        node = q.popleft()
        if node.right is not None and node.right in seen:
            return node.val
        for s in ("right", "left"):
            k = getattr(node, s)
            if k and k not in seen:
                seen.add(k); q.append(k)

def correct_left_defect(root):
    """Biến thể 1: lỗi right trỏ sang nút cùng tầng BÊN TRÁI -> duyệt trái trước."""
    seen = {root}; q = deque([(root, None, None)])
    while q:
        node, par, side = q.popleft()
        if node.right is not None and node.right in seen:
            setattr(par, side, None); return root
        for s in ("left", "right"):
            k = getattr(node, s)
            if k and k not in seen:
                seen.add(k); q.append((k, node, s))
    return root

print("nút lỗi ví dụ 2:", find_invalid(make_defect(*examples[1][:3])))
r = make_defect([1, 2, 3, 4, 5, None, 6], 6, 4)          # 6 (bên phải) trỏ sang 4 (bên trái), 6.right ban đầu None
print("biến thể trỏ sang trái:", to_list(correct_left_defect(r)), "| kỳ vọng:", expected_after_fix([1, 2, 3, 4, 5, None, 6], 6))

nút lỗi ví dụ 2: 7
biến thể trỏ sang trái: [1, 2, 3, 4, 5] | kỳ vọng: [1, 2, 3, 4, 5]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tầng + list | tính tầng, kiểm từng nút | `O(n·w)` | `O(n)` | hiểu bài, đối chiếu |
| 2. BFS + tập tầng | `u.right ∈ tầng hiện tại` | `O(n)` | `O(w)` | không phụ thuộc thứ tự trong tầng |
| 3. BFS phải → trái | `u.right ∈ seen` | `O(n)` | `O(n)` | **nộp bài**, lặp |
| 4. DFS đệ quy phải trước | trả `None` để cắt | `O(n)` | `O(n)` + stack | code ngắn nhất, cây nông |
| 5. DFS lặp phải trước | stack + (cha, phía) | `O(n)` | `O(n)` | cây sâu, thích DFS |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Cây `[1,2,3,4,5,6,7]` (đầy đủ 3 tầng) với lỗi `fromNode = 5`... (5.right ban đầu là `None`) trỏ tới `toNode = 7`. Kết quả level-order là gì?

**Bài 2 (tính tay):** Trace BFS phải → trái cho cây `[1,2,3,4,null,5,6]`, lỗi `4 → 6`: ghi thứ tự xử lý từng tầng và thời điểm phát hiện.

**Bài 3 (điền TODO):** Hoàn thiện `invalid_and_removed(root)` trả về `(giá trị nút lỗi, danh sách giá trị các nút bị xóa)` trước khi xóa.

**Bài 4 (tự viết lại):** Viết lại cách 4 (DFS đệ quy) dưới dạng lặp với stack nhưng **giữ phong cách "trả về nút mới"**: không lưu `(cha, phía)`, mà xử lý con trỏ con ngay khi duyệt cha.

**Bài 5 (mini challenge):** Viết `count_paths_to(root, val)` đếm **số đường đi khác nhau** từ gốc tới nút có giá trị `val` trong cây **lỗi** (DAG), bằng DP theo thứ tự topo (theo tầng, trái → phải); kiểm rằng đúng nút `toNode` và các hậu duệ của nó có 2 đường.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def invalid_and_removed(root):
    # TODO: tìm nút lỗi như Solution; trước khi cắt, duyệt cây con (chỉ qua left, vì right là con trỏ lỗi) để gom giá trị
    pass

# Bài 4 — TODO
def correct_iter_return_style(root):
    # TODO: stack các nút; khi xử lý node: nếu node.right là nút lỗi... (gợi ý: kiểm tra con trước khi đẩy)
    pass

# Bài 5 — TODO
def count_paths_to(root, val):
    # TODO: ways[root] = 1; duyệt theo tầng phải->trái, cộng ways[u] vào ways[child] cho mọi con trỏ (kể cả lỗi)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `5` là con phải của `2` — nhưng đề bảo đảm `fromNode.right` ban đầu là `None`; ở đây `5` là lá nên `5.right = None` ✔.
- Mức 2: Xóa `5` (lá) khỏi cha `2`.
- Mức 3: `[1,2,3,4,null,6,7]`.

**Bài 2**
- Mức 1: Tầng 1 xử lý `3` trước `2`.
- Mức 2: Tầng 2 hàng đợi `6, 5, 4`.
- Mức 3: Tại `4`: `4.right = 6 ∈ seen` ⇒ cắt `2.left`.

**Bài 3**
- Mức 1: Nút lỗi có `right` là con trỏ lỗi — đừng đi theo nó.
- Mức 2: Cây con thật của nút lỗi = nút lỗi + cây con `left`.
- Mức 3: Kiểm: tập bị xóa = tập nút của cây gốc trừ tập nút của cây kết quả.

**Bài 4**
- Mức 1: Trước khi đẩy con `c` vào stack, kiểm tra `c` có phải nút lỗi không (`c.right in seen`)…
- Mức 2: …nhưng `seen` phải đủ: cần thứ tự "phải trước" và đánh dấu khi **lấy ra**.
- Mức 3: Cách đơn giản: kiểm tra ở cha: nếu `node.left` / `node.right` là lỗi thì gán `None` ngay khi lấy `node` ra (sau khi đã thăm mọi nút bên phải).

**Bài 5**
- Mức 1: Cây lỗi là DAG — không có chu trình.
- Mức 2: Theo tầng, mọi cạnh đi từ tầng `d` xuống `d + 1`, trừ cạnh lỗi đi **ngang sang phải** ⇒ xử lý các tầng từ trên xuống, trong mỗi tầng **từ trái sang phải** là một thứ tự topo (nút lỗi được xử lý trước nút bị trỏ tới).
- Mức 3: `toNode` có 2 đường; mỗi hậu duệ của nó cũng có 2.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1
r1 = to_list(Solution().correctBinaryTree(make_defect([1, 2, 3, 4, 5, 6, 7], 5, 7)))
print("Bài 1:", r1)
assert r1 == [1, 2, 3, 4, None, 6, 7]

# Bài 2
trace_bfs_rtl([1, 2, 3, 4, None, 5, 6], 4, 6)

Bài 1: [1, 2, 3, 4, None, 6, 7]
tầng 0: hàng đợi = [1]
   xử lý 1: right = 3 chưa thăm -> thêm [3, 2]
tầng 1: hàng đợi = [3, 2]
   xử lý 3: right = 6 chưa thăm -> thêm [6, 5]
   xử lý 2: right = None chưa thăm -> thêm [4]
tầng 2: hàng đợi = [6, 5, 4]
   xử lý 6: right = None chưa thăm -> thêm []
   xử lý 5: right = None chưa thăm -> thêm []
   xử lý 4: right = 6 ĐÃ THĂM -> 4 là nút lỗi; cắt 2.left
   kết quả: [1, 2, 3, None, None, 5, 6]


TreeNode(1)

In [25]:
# Bài 3
def invalid_and_removed_sol(root):
    seen = {root}; q = deque([root])
    bad = None
    while q and bad is None:
        node = q.popleft()
        if node.right is not None and node.right in seen:
            bad = node; break
        for s in ("right", "left"):
            k = getattr(node, s)
            if k and k not in seen:
                seen.add(k); q.append(k)
    removed, stack = [], [bad]
    while stack:
        u = stack.pop(); removed.append(u.val)
        for k in ((u.left,) if u is bad else (u.left, u.right)):
            if k: stack.append(k)
    return bad.val, sorted(removed)

rnd = random.Random(3)
done = 0
while done < 600:
    vals_ = random_tree_values(rnd.randint(3, 20), rnd)
    pick = pick_defect(vals_, rnd)
    if not pick: continue
    v, removed = invalid_and_removed_sol(make_defect(vals_, *pick))
    after = [x for x in expected_after_fix(vals_, pick[0]) if x is not None]
    before = [x for x in vals_ if x is not None]
    assert v == pick[0] and sorted(set(before) - set(after)) == removed
    done += 1
print("Bài 3 OK | ví dụ 2:", invalid_and_removed_sol(make_defect(*examples[1][:3])))

Bài 3 OK | ví dụ 2: (7, [2, 7])


In [26]:
# Bài 4
def correct_iter_return_style_sol(root):
    seen = {root}
    q = deque([root])                 # BFS phải->trái: kiểm tra con TRƯỚC khi đưa vào
    while q:
        node = q.popleft()
        for s in ("right", "left"):
            c = getattr(node, s)
            if c is None:
                continue
            # c là nút lỗi nếu c.right đã thăm. Ở thời điểm này mọi nút cùng tầng bên phải c
            # đều đã được đưa vào seen (vì cha của chúng ở bên phải node, hoặc là node với phía right).
            if c.right is not None and c.right in seen:
                setattr(node, s, None)
                return root
            if c not in seen:
                seen.add(c); q.append(c)
    return root

rnd = random.Random(4)
done = 0
while done < 800:
    vals_ = random_tree_values(rnd.randint(3, 25), rnd)
    pick = pick_defect(vals_, rnd)
    if not pick: continue
    assert to_list(correct_iter_return_style_sol(make_defect(vals_, *pick))) == expected_after_fix(vals_, pick[0])
    done += 1
print("Bài 4 OK")

# Bài 5
def count_paths_to_sol(root, val):
    ways = {root: 1}
    level = [root]
    while level:
        nxt = []
        for u in level:                         # trái -> phải: from (bên trái) xử lý trước to => thứ tự topo
            for c in (u.left, u.right):
                if c is None: continue
                if c not in ways:
                    ways[c] = 0
                    if not (c in level):        # cạnh lỗi đi ngang: c thuộc cùng tầng, không thêm vào tầng sau
                        nxt.append(c)
                ways[c] += ways[u]
        level = nxt
    return next((w for node, w in ways.items() if node.val == val), 0)

r5 = make_defect(*examples[1][:3])
res5 = {v: count_paths_to_sol(r5, v) for v in [8, 3, 1, 7, 9, 4, 2, 5, 6]}
print("Bài 5: số đường đi:", res5)
assert res5[4] == 2 and res5[5] == 2 and res5[6] == 2 and res5[7] == 1

Bài 4 OK
Bài 5: số đường đi: {8: 1, 3: 1, 1: 1, 7: 1, 9: 1, 4: 2, 2: 1, 5: 2, 6: 2}


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template BFS phải → trái phát hiện "con trỏ ngang sang phải":**

```python
seen = {root}
q = deque([(root, None, None)])           # (nút, cha, phía)
while q:
    node, par, side = q.popleft()
    if node.right is not None and node.right in seen:
        setattr(par, side, None)          # xóa nút lỗi + cây con
        return root
    for s in ("right", "left"):           # PHẢI trước
        c = getattr(node, s)
        if c is not None and c not in seen:
            seen.add(c)
            q.append((c, node, s))
return root
```

**Template DFS (cây nông):** `if node is None or node.right in seen: return None; seen.add(node); node.right = dfs(node.right); node.left = dfs(node.left); return node`.

**Checklist:**
- [ ] Duyệt **phải trước** (BFS và DFS)?
- [ ] Xóa **nút lỗi** khỏi cha (không chỉ cắt con trỏ lỗi)?
- [ ] Có `seen` để không thăm `toNode` hai lần?
- [ ] Đánh dấu `seen` khi đưa vào hàng đợi (BFS)?
- [ ] Cây sâu `10⁴` nút: dùng bản lặp?
- [ ] Đã test bằng driver gây lỗi + đáp án chuẩn từ cây gốc?
- [ ] Chữ ký đúng: `def correctBinaryTree(self, root: TreeNode) -> TreeNode`?